In [1]:
#| default_exp frida

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

/usr/local/lib/python3.10/dist-packages/nbdev/export.py:80: UserWarning: Notebook '/workspaces/gpt/rugptxl_converter.ipynb' uses `#|export` without `#|default_exp` cell.
Note nbdev2 no longer supports nbdev1 syntax. Run `nbdev_migrate` to upgrade.
See https://nbdev.fast.ai/getting_started.html for more information.
  warn(f"Notebook '{nbname}' uses `#|export` without `#|default_exp` cell.\n"


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [4]:
#| export
from os import getenv
model_path = getenv("MODEL")
from rest.gen import generate


In [5]:
model_path = 'fred'

In [6]:
#| export
from optimum.onnxruntime import ORTModelForSeq2SeqLM

In [7]:
#| export
seq_length = 1024

full_path = f'./models/{model_path}'
import torch
from transformers import GPT2Tokenizer, T5ForConditionalGeneration, AutoTokenizer

In [8]:
def convert_and_save_model(model_path, save_dir):
    model = ORTModelForSeq2SeqLM.from_pretrained(model_path, export=True)
    model.save_pretrained(save_dir)
    
    # Also save the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    tokenizer.save_pretrained(save_dir)
    
    print(f"Model and tokenizer saved to {save_dir}")

In [9]:
#convert_and_save_model(full_path, full_path+'/optimized')

In [10]:
#| export
tokenizer = GPT2Tokenizer.from_pretrained(full_path+'/optimized/', eos_token='</s>')
model = ORTModelForSeq2SeqLM.from_pretrained(full_path+'/optimized/', provider="CUDAExecutionProvider")

2024-09-30 11:57:48.483629382 [W:onnxruntime:, session_state.cc:1166 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2024-09-30 11:57:48.483653821 [W:onnxruntime:, session_state.cc:1168 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2024-09-30 11:57:49.164057460 [W:onnxruntime:, session_state.cc:1166 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2024-09-30 11:57:49.164073829 [W:onnxruntime:, session_state.cc:1168 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2024-09-30 11:57:49.999209709 [W:onnxrun

In [16]:
#| export
def iftoken(tokenizer, tokens):
    # returns token id if the given string is one token
    token_ids = [tokenizer.encode(token, add_special_tokens=False) for token in tokens]
    return [id for sublist in token_ids for id in sublist if len(sublist) == 1]



In [22]:
#| export
from front.common import process_seq

def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool, temperature:float=0.5):
    blocked_tokens = ['[', '(', '\xa0', '*', '­', '~', '_', '\\', '\uf04a', '\ufeff', '\u2028']
    if not allow_linebreak:
        blocked_tokens.extend(['\n', '\n\n',' \n'])
    bad_words_ids = iftoken(tokenizer, blocked_tokens)
    bad_words_ids = [[w] for w in bad_words_ids]
    
    lm_text = '<LM>' + prompt
    input_ids=torch.tensor([tokenizer.encode(lm_text)]).cuda()
    output_ids = model.generate(input_ids, do_sample=True, temperature=temperature, repetition_penalty=5.0, typical_p=0.9, top_k=10, top_p=0.95, #watermark=False,
                        max_new_tokens=length, bad_words_ids = bad_words_ids,
                        num_return_sequences=num_samples,)

    result = [tokenizer.decode(o[1:]).replace('\n', ' ') for o in output_ids]
    result = process_seq(result)
    return result


In [23]:
%%time
get_sample('<LM>На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.25 s, sys: 26.2 ms, total: 1.28 s
Wall time: 642 ms


[' – просто свинья. И даже не потому, что ты нагадил в штаны и обосрался с перепугу».',
 ' – говно. Вот и все, что я могу сказать». Но тут же вспомнил о своей новой работе в «Ночном дозоре», которая была для него важнее всего остального мира: ведь он мог быть там единственным человеком с живым сердцем!',
 ' — говно. И я не знаю, что хуже». Я даже подумал: «А вдруг и правда?» Но потом понял все-таки другое… Что это просто такая игра слов? А на самом деле он действительно говнюк!',
 ' – просто говно. Но это, конечно же не так». Я ответил: «Да ладно тебе!']

In [24]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.32 s, sys: 15.5 ms, total: 1.33 s
Wall time: 580 ms


[' – просто свинья. И не надо мне тут про «святую Русь». Я, может быть и русский человек в душе (а ты знаешь кто такой на самом деле этот твой Достоевский?',
 ' – просто мудак. И все, что ты написал за всю свою жизнь в своих книгах и статьях о любви к ближнему своему (а это очень много), не стоит ни гроша». Вот так-то!',
 ' – просто говно. Ты даже не знаешь, как это звучит по-английски».',
 ' – просто говно. А вот если бы ты был, к примеру… как это называется? Ну да! Ты стал балериной».']

In [25]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.35 s, sys: 0 ns, total: 1.35 s
Wall time: 747 ms


[' – просто говно. Но, может быть… Может ведь такое случиться? Ты не думай!',
 ' – говно. И это не только про меня, но и вообще». А потом я понял: он прав!',
 ' — говно». Я не стал спорить, но и ничего нового для себя тоже.',
 ' – просто говно». Но я не стал с ним спорить.']